# 07 · Análisis estadístico
**Responsables:** P5 (librería e hipótesis principal), P2 (hipótesis A), P4 (hipótesis B) · **Código:** `src/estadistica.py` · **Salidas:** `outputs/stats/`

**Plan fijado antes de ver los resultados** (no se cambia según lo que salga)
- α = 0,05. Hipótesis principal unilateral: H1 = asociación **positiva** entre `prop_18_44_pct` y `puntaje_mdm` (2020, hasta 123 municipios).
- Pearson y Spearman con IC 95 % y valor p. Spearman es la principal si Shapiro-Wilk rechaza normalidad o si hay influyentes.
- Regresión simple y con controles: log de la población, % rural y grupo de capacidades del MDM (G1–G6). Breusch-Pagan → errores HC3 si hay heterocedasticidad. Distancia de Cook > 4/n para influyentes. VIF.
- Sensibilidad: sin influyentes, con el bloque alterno del MDM y con la proporción calculada desde la fuente por áreas.
- Complementarias (exploratorias, p ajustado por Holm): A = producción vs población 18–64; B = % de suelo III–IV vs rendimiento de papa; D = mujeres y hombres 18–44 vs desempeño.

**Cómo concluir:** "hay evidencia de asociación positiva", "no se rechaza H0" o "datos insuficientes", siempre con la magnitud (r) y su intervalo. **Nunca** causalidad ni "disposición al cambio".

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [2]:
import pandas as pd
from src import estadistica as est, graficos
from src.silver_poblacion import prop_18_44_desde_areas

ma = pl.read_parquet(GOLD / "municipio_anio.parquet")
d2020 = ma.filter(pl.col("anio") == ANIO_MDM).to_pandas()
d2023 = ma.filter(pl.col("anio") == ANIO_EVA).to_pandas()

mdm = pl.read_parquet(SILVER / "desempeno.parquet")
alt_mdm = mdm.filter(~pl.col("es_bloque_oficial")).select("codigo_municipio", puntaje_mdm_alt="puntaje_mdm")
alt_pob = prop_18_44_desde_areas(pl.read_parquet(SILVER / "poblacion_area.parquet")).filter(pl.col("anio") == ANIO_MDM)
alterno = alt_mdm.join(alt_pob.drop("anio"), on="codigo_municipio", how="full", coalesce=True).to_pandas()

## Hipótesis principal (P5)

In [3]:
principal = est.hipotesis_principal(d2020, alterno)
principal[["analisis", "metodo", "n", "r", "ic95_inf", "ic95_sup", "p", "conclusion"]]

,analisis,metodo,n,r,ic95_inf,ic95_sup,p,conclusion
0,principal,pearson,123,0.251874,0.078333,0.410600,0.002474,Hay evidencia de asociación positiva
1,principal,spearman,123,0.240690,0.063927,0.402810,0.003663,Hay evidencia de asociación positiva
2,sensibilidad_sin_influyentes,pearson,119,0.269964,0.094564,0.429108,0.001493,Hay evidencia de asociación positiva
3,sensibilidad_sin_influyentes,spearman,119,0.246955,0.067334,0.411071,0.003389,Hay evidencia de asociación positiva
4,sensibilidad_bloque_alterno_mdm,spearman,123,0.241682,0.064955,0.403710,0.003540,Hay evidencia de asociación positiva
5,sensibilidad_poblacion_fuente_alterna,spearman,123,0.233919,0.056919,0.396662,0.004607,Hay evidencia de asociación positiva


In [4]:
pd.read_csv(STATS / "h_principal_normalidad.csv")

,variable,n,shapiro_w,p,normal_al_5pct,asimetria,curtosis
0,prop_18_44_pct,123,0.988253,0.373837,True,0.307079,0.596308
1,puntaje_mdm,123,0.981853,0.097230,True,-0.379580,-0.369782


In [5]:
print((STATS / "regresion_h_principal_controles_resumen.txt").read_text(encoding="utf-8"))

                            OLS Regression Results                            
Dep. Variable:            puntaje_mdm   R-squared:                       0.160
Model:                            OLS   Adj. R-squared:                  0.108
Method:                 Least Squares   F-statistic:                     3.119
Date:                Thu, 05 Nov 2026   Prob (F-statistic):            0.00473
Time:                        12:54:09   Log-Likelihood:                -417.84
No. Observations:                 123   AIC:                             851.7
Df Residuals:                     115   BIC:                             874.2
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

## Hipótesis complementarias (P2, P4) y consolidado con Holm

In [6]:
a = est.hipotesis_produccion_poblacion(d2023)   # P2
b = est.hipotesis_suelo_rendimiento(d2023)      # P4
d = est.hipotesis_sexo(d2020)                   # P5
todo = est.consolidar(principal, [a, b, d])
todo[["tipo", "analisis", "metodo", "n", "r", "p", "p_holm", "conclusion"]]

,tipo,analisis,metodo,n,r,p,p_holm,conclusion
0,principal,principal,pearson,123,0.251874,2.474325e-03,NaN,Hay evidencia de asociación positiva
1,principal,principal,spearman,123,0.240690,3.662761e-03,NaN,Hay evidencia de asociación positiva
2,principal,sensibilidad_sin_influyentes,pearson,119,0.269964,1.492823e-03,NaN,Hay evidencia de asociación positiva
3,principal,sensibilidad_sin_influyentes,spearman,119,0.246955,3.388611e-03,NaN,Hay evidencia de asociación positiva
4,principal,sensibilidad_bloque_alterno_mdm,spearman,123,0.241682,3.539988e-03,NaN,Hay evidencia de asociación positiva
5,principal,sensibilidad_poblacion_fuente_alterna,spearman,123,0.233919,4.606761e-03,NaN,Hay evidencia de asociación positiva
6,exploratoria,A_produccion_vs_pob_18_64,spearman,123,0.502425,3.195886e-09,1.597943e-08,Hay evidencia de asociación
7,exploratoria,A_log_produccion_vs_log_pob,pearson,123,0.424375,9.995092e-07,3.998037e-06,Hay evidencia de asociación
8,exploratoria,B_suelo_iii_iv_vs_rendimiento_papa,spearman,92,0.408575,5.270870e-05,1.581261e-04,Hay evidencia de asociación
9,exploratoria,D_mujeres_18_44_vs_mdm,spearman,123,0.327063,2.221882e-04,4.443765e-04,Hay evidencia de asociación


## Dispersiones (P5)

In [7]:
fila = principal[(principal.analisis == "principal") & (principal.metodo == "spearman")].iloc[0].to_dict()
graficos.dispersion(d2020, "prop_18_44_pct", "puntaje_mdm",
                    f"Población de 18-44 años vs desempeño municipal, {ANIO_MDM}",
                    "Población de 18 a 44 años (% del total)", "Puntaje MDM (0-100)",
                    "DANE, proyecciones; DNP, Medición del Desempeño Municipal", "p5_dispersion_joven_mdm", fila)
fa = a.iloc[0].to_dict()
graficos.dispersion(d2023, "pob_18_64", "produccion_total_t",
                    f"Producción agrícola vs población de 18-64 años, {ANIO_EVA}",
                    "Población de 18 a 64 años (personas, escala log)", "Producción total (t, escala log)",
                    "UPRA - EVA; DANE - proyecciones", "p5_dispersion_produccion_poblacion", fa, log=True)

PosixPath('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/proyecto_boyaca/outputs/figures/p5_dispersion_produccion_poblacion.png')